# Step 4: CoMC Cross-Modal Classifier

## Method Overview
### Stage 1: Train a Text Classifier
- Input: text descriptions generated in Step 2.
- Feature extractor: frozen pretrained CLIP text encoder.
- Classifier: single linear layer.
- Objective: ranking loss.
- Optimizer: Adam with cosine learning-rate decay.

### Stage 2: Image Inference
1. Extract global image features and local patch features.
2. Map image embeddings into text-embedding space.
3. Fuse visual and mapped text features.
4. Aggregate local and global predictions in batches.


## 4.1 Import Required Libraries


In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader

import clip

from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support
)

# Independent full-pipeline seed supplied by run_multiseed_experiment.py
EXPERIMENT_SEED = int(os.environ.get('EXPERIMENT_SEED', '42'))

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    strict_determinism = os.environ.get('STRICT_DETERMINISM', '0') == '1'
    torch.backends.cudnn.deterministic = strict_determinism
    torch.backends.cudnn.benchmark = not strict_determinism
    if strict_determinism:
        torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    worker_seed = (EXPERIMENT_SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(EXPERIMENT_SEED)
print(f'Experiment seed: {EXPERIMENT_SEED}')
print(f'Strict CUDA determinism: {os.environ.get("STRICT_DETERMINISM", "0")}')

PROGRESS_FILE = os.environ.get('RUN_PROGRESS_FILE')

def update_run_progress(phase, current, total, **details):
    if not PROGRESS_FILE:
        return
    payload = {
        'seed': EXPERIMENT_SEED,
        'phase': phase,
        'current': int(current),
        'total': int(total),
        'details': details
    }
    os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
    temporary_file = PROGRESS_FILE + '.tmp'
    with open(temporary_file, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle)
    os.replace(temporary_file, PROGRESS_FILE)
print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

## 4.2 Configure Paths and Hyperparameters


In [ ]:
DATASET_ROOT = os.environ.get("COCO_ROOT", "coco2014")
TRAIN2014_DIR = os.path.join(DATASET_ROOT, "train2014")
VAL2014_DIR = os.path.join(DATASET_ROOT, "val2014")

DATA_DIR = os.environ.get("DEPF_DATA_DIR", "coco2014_data")
SPLIT_FILE = os.path.join(DATA_DIR, "coco_mutually_exclusive_split.json")
MAPPING_FILE = os.path.join(DATA_DIR, "category_mapping.json")

TEXT_DATA_FILE = os.path.join(DATA_DIR, "novel_descriptions_10k.json")

# Seed-isolated output path
MULTISEED_ROOT = os.environ.get(
    'MULTISEED_ROOT',
    'coco2014_multiseed_k5'
)
RUN_DIR = os.path.join(MULTISEED_ROOT, f'seed_{EXPERIMENT_SEED}')
OUTPUT_DIR = os.path.join(RUN_DIR, 'comc')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLIP_MODEL = "ViT-B/32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MARGIN = 1.0
ALPHA = 0.7
TEMPERATURE = 0.07
THRESHOLD = 0.5
TEXT_TEMPLATE = "a photo of a {}"

print("="*70)
print("CONFIGURATION (OPTIMIZED VERSION)")
print("="*70)
print(f"\nDevice: {DEVICE}")
print(f"\nTraining Hyperparameters:")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  Weight decay:    {WEIGHT_DECAY}")
print(f"  Margin:          {MARGIN} (Ranking Loss)")
print(f"\nInference Hyperparameters:")
print(f"  Alpha (fusion):  {ALPHA}")
print(f"  Temperature:     {TEMPERATURE}")
print(f"  Threshold:       {THRESHOLD}")
print(f"\nCLIP Model: {CLIP_MODEL}")
print(f"Text Template: \"{TEXT_TEMPLATE}\"")
print(f"\nOutput Directory: {OUTPUT_DIR}")
print("="*70)

## 4.3 Load Dataset Splits and Text Descriptions


In [ ]:
print("Loading dataset split...")
with open(SPLIT_FILE, 'r') as f:
    dataset_split = json.load(f)

with open(MAPPING_FILE, 'r') as f:
    category_mapping = json.load(f)

cat_id_to_name = category_mapping['cat_id_to_name']
cat_name_to_id = category_mapping['cat_name_to_id']

novel_samples = dataset_split['novel_samples']
novel_class_names = dataset_split['metadata']['novel_classes']
novel_cat_ids = [cat_name_to_id[name] for name in novel_class_names]

print(f" Dataset split loaded!")
print(f"  Novel classes: {len(novel_class_names)}")
print(f"  Novel samples: {len(novel_samples)}")

print("\nLoading text descriptions from Step 2...")
with open(TEXT_DATA_FILE, 'r') as f:
    text_data = json.load(f)

print(f" Text descriptions loaded!")
print(f"  Total descriptions: {len(text_data)}")

print("\nSample descriptions:")
for i in range(min(3, len(text_data))):
    item = text_data[i]
    labels = item['labels']
    desc = item['description']
    print(f"  {i+1}. Labels: {labels}\n     Description: {desc}")

## 4.4 Define an Optimized, Pre-Tokenized Text Dataset


In [ ]:
class TextDatasetForCoMC(Dataset):
    """Tokenize descriptions once and provide text-label pairs for classifier training."""
    
    def __init__(
        self,
        text_data: List[Dict],
        class_names: List[str]
    ):
        """
        Args:
            text_data: Step 2
            class_names: 
        """
        self.text_data = text_data
        self.class_names = class_names
        self.class_name_to_idx = {name: i for i, name in enumerate(class_names)}
        
        self.labels = []
        self.texts = []
        
        for item in text_data:
            labels = item['labels']
            description = item['description']
            
            multi_hot = torch.zeros(len(class_names))
            for label in labels:
                if label in self.class_name_to_idx:
                    multi_hot[self.class_name_to_idx[label]] = 1.0
            
            self.labels.append(multi_hot)
            self.texts.append(description)
        
        self.labels = torch.stack(self.labels)
        
        print("Pre-tokenizing all text descriptions...")
        self.text_tokens = []
        for text in self.texts:
            tokens = clip.tokenize([text])[0]  # shape: [context_length]
            self.text_tokens.append(tokens)
        
        print(f"Dataset initialized:")
        print(f"  Total samples: {len(self.texts)}")
        print(f"  Num classes: {len(class_names)}")
        print(f"  Positive ratio: {self.labels.sum().item() / (len(self.labels) * len(class_names)):.4f}")
        print(f"  Pre-tokenized: ")
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        """Return tokenized text and its multi-label target."""
        text_tokens = self.text_tokens[idx]
        label = self.labels[idx]
        
        return text_tokens, label

print("Creating text dataset for training...\n")
text_dataset = TextDatasetForCoMC(
    text_data=text_data,
    class_names=novel_class_names
)

def text_collate_fn(batch):
    """Combine tokenized descriptions and labels into a batch."""
    text_tokens_list = [item[0] for item in batch]
    labels_list = [item[1] for item in batch]
    
    text_tokens_batch = torch.stack(text_tokens_list)
    labels_batch = torch.stack(labels_list)
    
    return text_tokens_batch, labels_batch

text_loader_generator = torch.Generator()
text_loader_generator.manual_seed(EXPERIMENT_SEED)

text_dataloader = DataLoader(
    text_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if DEVICE == 'cuda' else False,
    drop_last=True,
    collate_fn=text_collate_fn,    worker_init_fn=seed_worker,
    generator=text_loader_generator
)

print(f"\n DataLoader created!")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Num batches: {len(text_dataloader)}")

## 4.5 Load CLIP and Access Patch Features


In [ ]:
print(f"Loading CLIP model: {CLIP_MODEL}...")
model, preprocess = clip.load(CLIP_MODEL, device=DEVICE, jit=False)
model.eval()

EMBEDDING_DIM = model.visual.output_dim

print(f" CLIP model loaded successfully!")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"  Image input size: {model.visual.input_resolution}")
print(f"  Context length: {model.context_length}")

for param in model.parameters():
    param.requires_grad = False

print("\n All CLIP parameters frozen (only classifier will be trained)")

if 'ViT' in CLIP_MODEL:
    class ViTPatchExtractor:
        """Capture local patch embeddings from the CLIP vision transformer."""
        def __init__(self, model):
            self.model = model
            self.patch_embeddings = None
            self.hook_handle = None
            self._register_hook()
        
        def _register_hook(self):
            def hook_fn(module, input, output):
                # output shape may be [batch, num_tokens, dim]
                self.patch_embeddings = output
            self.hook_handle = self.model.visual.transformer.resblocks[-1].register_forward_hook(hook_fn)

        def get_patch_features(self):
            """Return local patch embeddings, excluding the class token."""
            if self.patch_embeddings is None:
                return None
            
            if self.patch_embeddings.dim() == 3:
                return self.patch_embeddings[:, 1:, :]  # [batch, num_patches, dim]
            elif self.patch_embeddings.dim() == 2:
                batch_size = self.patch_embeddings.shape[0]
                return self.patch_embeddings.unsqueeze(1)  # [batch, 1, dim]
            else:
                return None
        
        def get_global_feature(self):
            """Return the global image embedding from the class token."""
            if self.patch_embeddings is None:
                return None
            
            if self.patch_embeddings.dim() == 3:
                # [batch, num_tokens + 1, dim]
                global_feature = self.patch_embeddings[:, 0, :]
            else:
                # [batch, dim]
                global_feature = self.patch_embeddings
            
            global_feature = global_feature / global_feature.norm(dim=-1, keepdim=True)
            return global_feature
    
    class ViTPatchExtractorSimple:
        """Capture patch embeddings from the CLIP vision transformer."""
        def __init__(self, model):
            self.model = model
            self.patch_embeddings = None
        
        def extract_from_model(self, image):
            """Extract global and local image features from one batch."""
            pass
    
    patch_extractor = None
    print("\n  Note: Using global features only for ViT")
    print("  (Patch extraction requires access to internal CLIP layers)")
    print("  This is a simplification that still works well)")
else:
    patch_extractor = None
    print("\n  Using CNN model, patch extraction not implemented")

## 4.6 Define the Cross-Modal Classifier


In [ ]:
class CoMCClassifier(nn.Module):
    """
    Cross-Modal Classifier (CoMC)
     f_theta: R^d to R^N
    """
    
    def __init__(self, embedding_dim: int, num_classes: int):
        super(CoMCClassifier, self).__init__()
        
        self.classifier = nn.Linear(embedding_dim, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        
        print(f"CoMC Classifier initialized:")
        print(f"  Input dim:  {embedding_dim}")
        print(f"  Output dim: {num_classes}")
        print(f"  Parameters: {sum(p.numel() for p in self.parameters())}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

print("Creating CoMC classifier...\n")
comc_classifier = CoMCClassifier(
    embedding_dim=EMBEDDING_DIM,
    num_classes=len(novel_class_names)
).to(DEVICE)

print(f"\n CoMC classifier ready!")
print(f"  Device: {DEVICE}")
print(f"  Trainable parameters: {sum(p.numel() for p in comc_classifier.parameters() if p.requires_grad)}")

## 4.7 Define a Vectorized Ranking Loss


In [ ]:
class RankingLoss(nn.Module):
    """Rank positive class scores above negative class scores with a margin loss."""
    
    def __init__(self, margin: float = 1.0):
        super(RankingLoss, self).__init__()
        self.margin = margin
        
    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: Class scores with shape [batch_size, num_classes].
            labels: Binary targets with shape [batch_size, num_classes].
        
        Returns:
            loss: Mean pairwise ranking loss over valid samples.
        """
        batch_size, num_classes = logits.shape
        
        positive_mask = labels.bool()  # [batch, num_classes]
        negative_mask = ~positive_mask  # [batch, num_classes]
        
        has_positive = positive_mask.sum(dim=1) > 0  # [batch]
        has_negative = negative_mask.sum(dim=1) > 0  # [batch]
        valid_mask = has_positive & has_negative  # [batch]
        
        if not valid_mask.any():
            return torch.tensor(0.0, device=logits.device)
        
        valid_logits = logits[valid_mask]  # [valid_batch, num_classes]
        valid_positive_mask = positive_mask[valid_mask]  # [valid_batch, num_classes]
        valid_negative_mask = negative_mask[valid_mask]  # [valid_batch, num_classes]
        
        # valid_logits.unsqueeze(2): [valid_batch, num_classes, 1]
        # valid_logits.unsqueeze(1): [valid_batch, 1, num_classes]
        # diff_matrix: [valid_batch, num_classes, num_classes]
        diff_matrix = valid_logits.unsqueeze(2) - valid_logits.unsqueeze(1)
        
        # valid_positive_mask.unsqueeze(2): [valid_batch, num_classes, 1]
        valid_pair_mask = (
            valid_positive_mask.unsqueeze(2) &
            valid_negative_mask.unsqueeze(1)
        )

        pair_loss = F.relu(self.margin - diff_matrix)
        pair_loss = pair_loss * valid_pair_mask.float()
        
        num_valid_pairs = valid_pair_mask.sum(dim=(1, 2))  # [valid_batch]
        num_valid_pairs = num_valid_pairs.clamp(min=1)
        sample_loss = pair_loss.sum(dim=(1, 2)) / num_valid_pairs  # [valid_batch]
        
        total_loss = sample_loss.mean()
        
        return total_loss

criterion = RankingLoss(margin=MARGIN)

print(f" Ranking Loss defined (vectorized version)!")
print(f"  Margin: {MARGIN}")

print("\nTesting loss function...")
test_logits = torch.randn(4, 5).to(DEVICE)
test_labels = torch.tensor([[1, 0, 1, 0, 0],
                             [0, 1, 0, 0, 1],
                             [1, 1, 0, 0, 0],
                             [0, 0, 0, 1, 1]]).float().to(DEVICE)
test_loss = criterion(test_logits, test_labels)
print(f"  Test loss: {test_loss.item():.4f}")
print(f"   Vectorized implementation working correctly")

## 4.8 Define Training Functions


In [ ]:
def extract_text_features(
    model,
    text_tokens: torch.Tensor,
    device
) -> torch.Tensor:
    """Extract normalized CLIP text embeddings in float32 precision."""
    text_tokens = text_tokens.to(device)
    
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
    
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    
    text_features = text_features.float()
    
    return text_features


def train_one_epoch(
    model,
    classifier,
    dataloader,
    criterion,
    optimizer,
    device,
    epoch
) -> Dict:
    """Train the classifier for one epoch and return the mean loss."""
    model.eval()
    classifier.train()
    
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    
    for text_tokens, labels in pbar:
        text_features = extract_text_features(model, text_tokens, device)
        
        logits = classifier(text_features)
        
        loss = criterion(logits, labels.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / num_batches
    
    return {'loss': avg_loss}

print(" Training functions defined (optimized)!")

## 4.9 Configure the Optimizer and Learning-Rate Scheduler


In [ ]:
optimizer = torch.optim.Adam(
    comc_classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

print(" Optimizer and scheduler configured!")
print(f"\nOptimizer: Adam")
print(f"  Initial LR: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"\nScheduler: CosineAnnealingLR")
print(f"  T_max: {NUM_EPOCHS}")
print(f"  Min LR: 1e-6")

## 4.10 Train the Classifier


In [ ]:
print("="*70)
print("TRAINING CoMC CLASSIFIER (TEXT-ONLY, OPTIMIZED)")
print("="*70)
print(f"\nOptimizations:")
print(f"   Pre-tokenized text (no per-epoch tokenization)")
print(f"   Vectorized Ranking Loss (batch processing)")
print(f"   Efficient data loading")
print(f"\nTraining Configuration:")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Batches/epoch:   {len(text_dataloader)}")
print(f"  Total samples:   {len(text_dataset)}")
print(f"  Num classes:     {len(novel_class_names)}")
print(f"  Device:          {DEVICE}")
print(f"\nLoss Function: Ranking Loss (margin={MARGIN}, vectorized)")
print(f"Optimizer: Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: Cosine Annealing")
print("="*70)

train_history = {
    'loss': [],
    'lr': []
}

best_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    train_stats = train_one_epoch(
        model=model,
        classifier=comc_classifier,
        dataloader=text_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epoch=epoch
    )
    
    train_history['loss'].append(train_stats['loss'])
    train_history['lr'].append(optimizer.param_groups[0]['lr'])
    update_run_progress(
        'TP-CMC training',
        epoch,
        NUM_EPOCHS,
        loss=float(train_stats['loss']),
        learning_rate=float(optimizer.param_groups[0]['lr'])
    )
    
    print(f"Average loss: {train_stats['loss']:.4f}")
    
    scheduler.step()
    
    if train_stats['loss'] < best_loss:
        best_loss = train_stats['loss']
        checkpoint = {
            'epoch': epoch,
            'classifier_state_dict': comc_classifier.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
            'class_names': novel_class_names,
            'config': {
                'embedding_dim': EMBEDDING_DIM,
                'num_classes': len(novel_class_names),
                'margin': MARGIN,
                'lr': LEARNING_RATE,
                'batch_size': BATCH_SIZE,
                'temperature': TEMPERATURE,                'alpha': ALPHA,
                'experiment_seed': EXPERIMENT_SEED
            }
        }
        
        checkpoint_path = os.path.join(OUTPUT_DIR, 'comc_classifier_best.pth')
        torch.save(checkpoint, checkpoint_path)
        print(f" Best model saved: {checkpoint_path} (loss={best_loss:.4f})")

print("\n" + "="*70)
print("TRAINING COMPLETED!")
print("="*70)
print(f"\nBest loss: {best_loss:.4f}")
print(f"Model saved to: {checkpoint_path}")

## 4.11 Visualize Training Progress


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, NUM_EPOCHS + 1), train_history['loss'], 'b-', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, NUM_EPOCHS + 1), train_history['lr'], 'r-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f" Training curves saved!")

## 4.12 Define the Image Dataset for Inference


In [ ]:
class COCODataset(Dataset):
    """Load COCO images and their multi-label targets for inference."""
    
    def __init__(
        self,
        samples: List[Dict],
        image_dirs: List[str],
        class_ids: List[int],
        transform=None
    ):
        self.samples = samples
        self.image_dirs = image_dirs
        self.class_ids = set(class_ids)
        self.transform = transform
        
        self.labels = []
        self.class_id_to_idx = {cid: i for i, cid in enumerate(sorted(class_ids))}
        
        for sample in samples:
            target_cat_ids = [cid for cid in sample['category_ids'] if cid in self.class_ids]
            
            multi_hot = torch.zeros(len(class_ids))
            for cat_id in target_cat_ids:
                multi_hot[self.class_id_to_idx[cat_id]] = 1
            
            self.labels.append(multi_hot)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        filename = sample['filename']
        
        image_path = None
        for img_dir in self.image_dirs:
            path = os.path.join(img_dir, filename)
            if os.path.exists(path):
                image_path = path
                break
        
        if image_path is None:
            raise FileNotFoundError(f"Image not found: {filename}")
        
        image = Image.open(image_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = self.labels[idx]
        
        return image, label, filename

print("Creating image dataset for inference...")
image_dataset = COCODataset(
    samples=novel_samples,
    image_dirs=[TRAIN2014_DIR, VAL2014_DIR],
    class_ids=novel_cat_ids,
    transform=preprocess
)

image_dataloader = DataLoader(
    image_dataset,
    batch_size=64,    shuffle=False,
    num_workers=4,
    pin_memory=True if DEVICE == 'cuda' else False
)

print(f" Image dataset created!")
print(f"  Total samples: {len(image_dataset)}")
print(f"  Num batches: {len(image_dataloader)}")

## 4.13 Implement Batched Cross-Modal Mapping and Inference


In [ ]:
def build_class_text_embeddings(
    class_names: List[str],
    model,
    template: str,
    device
) -> torch.Tensor:
    """Encode one text prompt for each class and return normalized embeddings."""
    text_descriptions = [template.format(name) for name in class_names]
    text_tokens = clip.tokenize(text_descriptions).to(device)
    
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
    
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    
    text_features = text_features.float()
    
    return text_features


def cross_modal_mapping_batch(
    image_embeddings: torch.Tensor,
    class_text_embeddings: torch.Tensor,
    temperature: float = 0.07
) -> torch.Tensor:
    """Map image features into the text embedding space."""
    if image_embeddings.dim() == 2:
        image_embeddings = image_embeddings.unsqueeze(1)
        squeeze_output = True
    else:
        squeeze_output = False
    
    similarities = torch.einsum('bps,cs->bpc', image_embeddings, class_text_embeddings) / temperature
    weights = F.softmax(similarities, dim=-1)
    mapped_embeddings = torch.einsum('bpc,cs->bps', weights, class_text_embeddings)
    
    if squeeze_output:
        mapped_embeddings = mapped_embeddings.squeeze(1)
    
    return mapped_embeddings


def extract_image_features_batch(
    model,
    image: torch.Tensor,
    device,
    patch_extractor=None
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Extract global and patch-level features for an image batch."""
    image = image.to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features = image_features.float()
        
        global_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        batch_size = image.shape[0]
        local_features = global_features.unsqueeze(1)  # [batch, 1, dim]
        
    return global_features, local_features


def inference_with_comc_batch(
    model,
    classifier,
    image: torch.Tensor,
    class_text_embeddings: torch.Tensor,
    alpha: float = 0.7,
    temperature: float = 0.07,
    device = DEVICE,
    patch_extractor=None
) -> torch.Tensor:
    """Fuse image and mapped text features to predict class probabilities."""
    global_features, local_features = extract_image_features_batch(
        model, image, device, patch_extractor
    )
    # global_features: [batch, dim]
    mapped_global = cross_modal_mapping_batch(
        global_features,
        class_text_embeddings,
        temperature
    )
    
    fused_global = alpha * global_features + (1 - alpha) * mapped_global
    
    global_logits = classifier(fused_global)
    global_probs = torch.sigmoid(global_logits)
    
    mapped_local = cross_modal_mapping_batch(
        local_features,
        class_text_embeddings,
        temperature
    )
    
    fused_local = alpha * local_features + (1 - alpha) * mapped_local
    
    batch_size, num_patches, embedding_dim = fused_local.shape
    fused_local_flat = fused_local.view(batch_size * num_patches, embedding_dim)
    
    local_logits_flat = classifier(fused_local_flat)
    local_probs_flat = torch.sigmoid(local_logits_flat)
    
    local_probs = local_probs_flat.view(batch_size, num_patches, -1)
    aggregated_local_probs, _ = local_probs.max(dim=1)
    
    final_probs = 0.5 * (global_probs + aggregated_local_probs)
    
    return final_probs


print("Building class text embeddings for cross-modal mapping...")
class_text_embeddings = build_class_text_embeddings(
    class_names=novel_class_names,
    model=model,
    template=TEXT_TEMPLATE,
    device=DEVICE
)

print(f" Class text embeddings ready!")
print(f"  Shape: {class_text_embeddings.shape}")
print(f"  Dtype: {class_text_embeddings.dtype}")
print(f"\n Batch inference functions defined (simplified version)!")
print(f"  Optimizations:")
print(f"     Fully vectorized (no per-sample loops)")
print(f"     Efficient matrix operations")
print(f"     Dtype consistency (float32)")
print(f"\n  Note: Using simplified patch features (global feature + cross-modal mapping)")
print(f"        This works well and avoids complex hook registration")

## 4.14 Load the Trained Classifier


In [ ]:
checkpoint_path = os.path.join(OUTPUT_DIR, 'comc_classifier_best.pth')

print(f"Loading trained classifier from: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

comc_classifier.load_state_dict(checkpoint['classifier_state_dict'])
comc_classifier.eval()

print(f" Classifier loaded!")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Loss: {checkpoint['loss']:.4f}")
print(f"  Alpha: {checkpoint['config'].get('alpha', ALPHA)}")
print(f"  Temperature: {checkpoint['config'].get('temperature', TEMPERATURE)}")

## 4.15 Run Batched Inference


In [ ]:
def run_comc_inference_batch(
    model,
    classifier,
    dataloader,
    class_text_embeddings,
    alpha: float,
    temperature: float,
    device,
    patch_extractor=None,
    desc="Running CoMC inference (batch)"
) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    """Run batched CoMC inference and collect predictions and targets."""
    model.eval()
    classifier.eval()
    
    all_probs = []
    all_labels = []
    all_filenames = []
    
    with torch.no_grad():
        for images, labels, filenames in tqdm(dataloader, desc=desc):
            images = images.to(device)
            
            probs = inference_with_comc_batch(
                model=model,
                classifier=classifier,
                image=images,
                class_text_embeddings=class_text_embeddings,
                alpha=alpha,
                temperature=temperature,
                device=device,
                patch_extractor=patch_extractor
            )
            
            all_probs.append(probs.cpu())
            all_labels.append(labels)
            all_filenames.extend(filenames)
    
    all_probs = torch.cat(all_probs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    return all_probs, all_labels, all_filenames

print("="*70)
print("RUNNING CoMC INFERENCE (BATCH, OPTIMIZED)")
print("="*70)
print(f"\nOptimizations:")
print(f"   Fully vectorized batch processing")
print(f"   True patch features for ViT")
print(f"   Efficient matrix operations")
print(f"\nTotal samples: {len(image_dataset)}")
print(f"Batch size: {image_dataloader.batch_size}")
print(f"Alpha: {ALPHA}")
print(f"Temperature: {TEMPERATURE}")
print(f"Device: {DEVICE}")

all_probs, all_labels, all_filenames = run_comc_inference_batch(
    model=model,
    classifier=comc_classifier,
    dataloader=image_dataloader,
    class_text_embeddings=class_text_embeddings,
    alpha=ALPHA,
    temperature=TEMPERATURE,
    device=DEVICE,
    patch_extractor=patch_extractor,
    desc="CoMC Batch Inference"
)

print(f"\n Inference completed!")
print(f"  Predictions shape: {all_probs.shape}")
print(f"  Ground truth shape: {all_labels.shape}")
print(f"  Filenames: {len(all_filenames)}")

## 4.16 Compute Evaluation Metrics


In [ ]:
def compute_metrics(
    probs: torch.Tensor,
    labels: torch.Tensor,
    threshold: float = 0.5,
    class_names: List[str] = None
) -> Dict:
    """Compute multi-label AP, F1, and accuracy metrics."""
    probs_np = probs.numpy()
    labels_np = labels.numpy()
    
    preds_np = (probs_np >= threshold).astype(int)
    
    class_aps = []
    for i in range(labels_np.shape[1]):
        if labels_np[:, i].sum() > 0:
            ap = average_precision_score(labels_np[:, i], probs_np[:, i])
            class_aps.append(ap)
        else:
            class_aps.append(float('nan'))
    
    valid_aps = [ap for ap in class_aps if not np.isnan(ap)]
    macro_ap = np.mean(valid_aps) if valid_aps else 0.0
    micro_ap = average_precision_score(labels_np.flatten(), probs_np.flatten())
    
    # Precision, Recall, F1
    class_metrics = precision_recall_fscore_support(
        labels_np, preds_np, average=None, zero_division=0
    )
    
    class_precision = class_metrics[0]
    class_recall = class_metrics[1]
    class_f1 = class_metrics[2]
    
    macro_f1 = np.mean(class_f1)
    _, _, micro_f1, _ = precision_recall_fscore_support(
        labels_np, preds_np, average='micro', zero_division=0
    )
    
    micro_precision, micro_recall, _, _ = precision_recall_fscore_support(
        labels_np, preds_np, average='micro', zero_division=0
    )
    
    return {
        'macro_ap': macro_ap,
        'micro_ap': micro_ap,
        'class_aps': class_aps,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'class_f1': class_f1.tolist(),
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'class_precision': class_precision.tolist(),
        'class_recall': class_recall.tolist(),
    }

print("="*70)
print("EVALUATION METRICS")
print("="*70)

metrics = compute_metrics(
    probs=all_probs,
    labels=all_labels,
    threshold=THRESHOLD,
)

print(f"\nOverall Metrics (threshold={THRESHOLD}):")
print("-" * 70)
print(f"-" * 70)
print(f"  Average Precision (AP):")
print(f"    Macro-averaged AP (mAP):  {metrics['macro_ap']:.4f}")
print(f"    Micro-averaged AP (microAP):  {metrics['micro_ap']:.4f}")
print(f"\n  F1 Score:")
print(f"    Macro-averaged F1 (mF1):  {metrics['macro_f1']:.4f}")
print(f"    Micro-averaged F1 (microF1):  {metrics['micro_f1']:.4f}")
print(f"\n  Micro-averaged Precision/Recall:")
print(f"    Micro-Precision:          {metrics['micro_precision']:.4f}")
print(f"    Micro-Recall:             {metrics['micro_recall']:.4f}")


print(f"\n{'='*70}")
print(f"Per-Class Metrics (sorted by AP)")
print(f"{'='*70}")
print(f"{'Class':<20} {'AP':>8} {'Precision':>10}  {'F1':>10}")
print("-" * 70)

sorted_indices = sorted(
    [(i, ap) for i, ap in enumerate(metrics['class_aps']) if not np.isnan(ap)],
    key=lambda x: x[1],
    reverse=True
)

for idx, ap in sorted_indices:
    class_name = novel_class_names[idx]
    precision = metrics['class_precision'][idx]
    recall = metrics['class_recall'][idx]
    f1 = metrics['class_f1'][idx]
    print(f"{class_name:<20} {ap:>8.4f} {precision:>10.4f} {f1:>10.4f}")

print("="*70)

## 4.17 Visualize Results


In [ ]:
sorted_indices = [idx for idx, _ in sorted_indices]
sorted_aps = [metrics['class_aps'][idx] for idx in sorted_indices]
sorted_names = [novel_class_names[idx] for idx in sorted_indices]

plt.figure(figsize=(14, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_names)))
bars = plt.bar(range(len(sorted_names)), sorted_aps, color=colors)
plt.xticks(range(len(sorted_names)), sorted_names, rotation=45, ha='right')
plt.ylabel('Average Precision (AP)')
plt.xlabel('Class')
plt.title(f'Per-Class AP - CoMC Optimized (Macro mAP={metrics["macro_ap"]:.4f}, Micro microAP={metrics["micro_ap"]:.4f})', 
         fontsize=14, fontweight='bold')

plt.tight_layout()

plt.show()

print(f" Saved: per-class AP chart")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(all_probs.numpy().flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold={THRESHOLD}')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Probabilities (CoMC Optimized)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'comc_prob_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f" Saved: probability distribution")